# Dynamic-k Comparison Notebook

Run this notebook top to bottom in the online IDE. It does not require opening a terminal. The cells call the project runner from Python, compare the baseline methods against the two stochastic dynamic-k methods, and display the saved results and plots.

The default path is conservative: first run a tiny smoke comparison. After that succeeds, set `RUN_REAL = True` in the real-run cell to use the default Qwen models on the 40 GB GPU. Smoke and real outputs are saved to separate folders.

In [1]:
from __future__ import annotations

import csv
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

# Set this only if your notebook kernel starts outside the repository.
REPO_ROOT_OVERRIDE = ""


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "main.py").exists() and (candidate / "research").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Set REPO_ROOT_OVERRIDE.")


REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "research" / "v.poponnikov" / "notebooks" / "dynamic_k_comparison.py"
RESULT_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "results"
PLOTS_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "plots"
RESULT_DIRS = {
    "smoke": RESULT_BASE_DIR / "smoke",
    "real": RESULT_BASE_DIR / "real",
}
PLOTS_DIRS = {
    "smoke": PLOTS_BASE_DIR / "smoke",
    "real": PLOTS_BASE_DIR / "real",
}
DISPLAY_RUN = "smoke"  # Change to "real" after the real run finishes.
EXPERIMENTS = ["01_baseline", "08_+speedup_adapt", "stochastic_consensus_k", "latent_regime_k"]

print(f"Repository: {REPO_ROOT}")
print(f"Comparison script: {SCRIPT}")

Repository: /home/jovyan/persistent_volume/Adaptive-speculative-decoding
Comparison script: /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/notebooks/dynamic_k_comparison.py


## Helper for Cell-Based Runs

This helper runs Python commands with `PYTHONPATH=src` and streams output into the notebook cell.

In [2]:
def run_python(*args: object) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src")
    env.setdefault("HF_HOME", str(REPO_ROOT / ".hf-cache"))
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("USE_TF", "0")
    env.setdefault("USE_FLAX", "0")
    env.setdefault("TRANSFORMERS_NO_TF", "1")
    env.setdefault("TRANSFORMERS_NO_FLAX", "1")
    env.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

    command = [sys.executable, *[str(arg) for arg in args]]
    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")

## Optional Dependency Install

Run this once on a fresh online IDE image. It installs runtime dependencies only, skipping the incompatible `dev` extra on Python 3.10. It also forces a CUDA 12.4 PyTorch build, which is compatible with servers whose NVIDIA driver reports CUDA 12.5. The NumPy pin is intentionally `<2` because this online image has older compiled packages such as TensorFlow, numexpr, and bottleneck.

In [3]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    run_python(
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
        "torch>=2.5,<2.11",
    )
    runtime_packages = [
        "typer>=0.15",
        "rich>=13",
        "tqdm>=4.68",
        "transformers>=4.48,<5",
        "accelerate>=1.3",
        "bitsandbytes>=0.45",
        "safetensors>=0.5",
        "datasets>=3,<5",
        "pandas>=2.2",
        "numpy<2",
        "scipy>=1.14",
        "scikit-learn>=1.6",
        "tokenizers>=0.21",
        "regex>=2024",
        "pyahocorasick>=2.3",
        "networkx>=3.4",
        "pydantic>=2.10",
        "python-dotenv>=1.2",
        "matplotlib>=3.10",
        "seaborn>=0.13",
    ]
    run_python("-m", "pip", "install", *runtime_packages)
    print("Dependency install finished. Restart the notebook kernel once, then rerun the setup/helper cells with INSTALL_DEPENDENCIES = False.")
else:
    print("Dependency install skipped.")

Running: /opt/conda/bin/python -m pip install --force-reinstall --index-url https://download.pytorch.org/whl/cu124 torch>=2.5,<2.11
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/24.6 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/24.6 MB 2.3 MB/s eta 0:00:11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/24.6 MB 2.2 MB/s eta 0:00:12
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/24.6 MB 2.3 MB/s eta 0:00:11
     ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/24.6 MB 3.2 MB/s eta 0:00:08
     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.7/24.6 MB 3.9 MB/s eta 0:00:07
     ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/24.6 MB 5.2 MB/s eta 0:00:05
     ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/24.6 MB 6.9 MB/s eta 0:00:04
     ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/24.6 MB 9.4 MB/s eta 0:00:03
     ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/24.6 MB 12.9 MB/s eta 0:00:02
     ━━━━━━━━━━╺━━━━━━━━━

## GPU Check

In [3]:
try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", props.name)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
except ModuleNotFoundError:
    print("Torch is not installed in this kernel. Enable the dependency install cell above.")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB MIG 3g.40gb
VRAM GB: 39.25


## Confirm Research Experiments Are Discoverable

In [4]:
run_python(REPO_ROOT / "src" / "main.py", "--list", "--research")

Running: /opt/conda/bin/python /home/jovyan/persistent_volume/Adaptive-speculative-decoding/src/main.py --list --research
Research experiments:

  latent_regime_k  — Stochastic dynamic k via latent regimes and change points
  stochastic_consensus_k  — Stochastic dynamic k via drafter self-consensus


## Tiny Smoke Comparison

This runs all four comparison experiments with small OPT models. Use this first to check that the online IDE can download models, load datasets, and write plots.

In [5]:
RUN_SMOKE = True
SMOKE_SAMPLES = 5
SMOKE_MAX_NEW_TOKENS = 32

if RUN_SMOKE:
    run_python(
        SCRIPT,
        "--output-dir",
        RESULT_DIRS["smoke"],
        "--plots-dir",
        PLOTS_DIRS["smoke"],
        "--tiny",
        "--samples",
        SMOKE_SAMPLES,
        "--max-new-tokens",
        SMOKE_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Smoke run skipped.")

Running: /opt/conda/bin/python /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/notebooks/dynamic_k_comparison.py --output-dir /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/results/dynamic_k_comparison_smoke --plots-dir /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/plots/dynamic_k_comparison_smoke --tiny --samples 5 --max-new-tokens 32 --device cuda
`torch_dtype` is deprecated! Use `dtype` instead!
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn

## Real Qwen Comparison

After the smoke run succeeds, set `RUN_REAL = True` and run this cell. With 40 GB VRAM, the default 0.5B drafter plus 7B target in 4-bit should fit.

In [6]:
RUN_REAL = True
REAL_SAMPLES = 50
REAL_MAX_NEW_TOKENS = 128

if RUN_REAL:
    run_python(
        SCRIPT,
        "--output-dir",
        RESULT_DIRS["real"],
        "--plots-dir",
        PLOTS_DIRS["real"],
        "--no-tiny",
        "--samples",
        REAL_SAMPLES,
        "--max-new-tokens",
        REAL_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Real run skipped. Set RUN_REAL = True when the smoke run is clean.")

Running: /opt/conda/bin/python /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/notebooks/dynamic_k_comparison.py --output-dir /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/results/dynamic_k_comparison_real --plots-dir /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/plots/dynamic_k_comparison_real --no-tiny --samples 50 --max-new-tokens 128 --device cuda
`torch_dtype` is deprecated! Use `dtype` instead!
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  w

## View Merged Results

In [7]:
csv_path = RESULT_DIRS[DISPLAY_RUN] / "dynamic_k_comparison.csv"
print(f"Displaying {DISPLAY_RUN} results from: {csv_path}")
if not csv_path.exists():
    print(f"No merged CSV yet: {csv_path}")
else:
    try:
        import pandas as pd

        display(pd.read_csv(csv_path))
    except ModuleNotFoundError:
        with csv_path.open("r", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                print(row)

Displaying smoke results from: /home/jovyan/persistent_volume/Adaptive-speculative-decoding/research/v.poponnikov/results/dynamic_k_comparison_smoke/dynamic_k_comparison.csv


,experiment,name,n_sequences,acceptance_rate,avg_accepted_tokens,avg_draft_length,cache_hit_rate,tokens_per_sec,avg_tokens_per_sec,total_new_tokens,...,regime_k_max_selected_k,regime_k_mean_step_acceptance,regime_k_k_distribution,regime_k_posterior_entropy_mean,regime_k_change_point_mean,regime_k_lambda_easy,regime_k_lambda_normal,regime_k_lambda_hard,regime_k_lambda_transition,regime_k_regime_distribution
0,01_baseline,01_baseline,5,0.212936,1.092308,5.000000,0.0,6.933344,7.097028,71,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,08_+speedup_adapt,08_+speedup_adapt,5,0.229988,1.210526,5.473684,0.0,7.734966,8.337184,92,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,stochastic_consensus_k,stochastic_consensus_k,5,0.579905,0.626506,1.168675,0.0,1.932143,2.130741,52,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,latent_regime_k,latent_regime_k,5,0.549658,1.777778,3.507937,0.0,10.364172,12.919917,112,...,8.0,0.487339,"{""1"": 17, ""2"": 12, ""3"": 3, ""4"": 8, ""5"": 10, ""6...",0.259112,0.163449,7.10334,5.076875,1.856761,1.210964,"{""easy"": 28, ""hard"": 8, ""normal"": 2, ""transiti..."


## View Plots

In [ ]:
plot_names = [
    "primary_metrics.png",
    "dynamic_k_summary.png",
    "dynamic_k_distribution.png",
    "controller_diagnostics.png",
]

print(f"Displaying {DISPLAY_RUN} plots from: {PLOTS_DIRS[DISPLAY_RUN]}")
for plot_name in plot_names:
    path = PLOTS_DIRS[DISPLAY_RUN] / plot_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot: {path}")